## 子词分词

### 目标
* 理解什么是子词分词
* 子词分词如何处理罕见词，未见词，表情符号
* 特殊标记的用途

In [ ]:
%%capture

# Install the custom package for this course. This also installs the gemma
# package.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

from gemma import gm # For interacting with the Gemma tokenizer.
# For providing feedback on your implementations.
from ai_foundations.feedback.course_2 import subword_tokens as feedback

## 子词分词
* 常用词作为完整标记
* 罕见或复杂词拆分为更小的子单元
* 未知的词拆分为字符或者其他的子词

In [ ]:
text = "等了五年，漫威这回好像真把大的憋出来了"
gemma_tokenizer  = gm.text.Gemma3Tokenizer()
gemma_tokens = gemma_tokenizer.encode(text)
print(f"{gemma_tokens}")
decoded_text = gemma_tokenizer.decode(gemma_tokens)
for token in gemma_tokens:
    decoded_token = gemma_tokenizer.decode(token)
    print(f"Token {token}:\t{decoded_token}")

## BPE算法
它从最小的单元——单个字符开始，迭代地将出现频率最高的相邻字符对合并成新的、更大的标记。
* 示例
 - deserts: 5 times
 - desert: 3 times
 - deserted: 2 times
 - tested: 6 times
* ，BPE 将每个单词拆分成单个字符，并添加一个特殊的词尾符号，“</w> ”，用于标记单词边界。这有助于模型区分单词内部的“er”（如“deserts”中的“er”）和单词末尾的“er”（如“tester”中的“er”）
* 现在，统计整个语料库中所有相邻词元对的数量，并考虑它们的频率。在序列“test”中 </w>” 标记对为 (t, e)、(e, s)、(s, t) 和 (t,</w> 这些配对计数汇总到频率表中
* 语料库中出现频率最高的词对是“es”，出现了 16 次。这对词对被合并成一个词“es”，并将这个新词添加到词汇表中。新的词汇表为{“d”, “e”, “s”, “r”, “t”, “</w> ”，“es”}。
* 数据集中所有出现的标记对“es”都被替换为新的标记“es”，从而得到以下更新后的数据集
 - d es e r t s </w> (5 times)
 - d es e r t </w> (3 times)
 - d es e r t e d </w> (2 times)
 - t es t e d </w> (6 times)
* 使用更新后的语料库，重复步骤 2 和 3。计算词对数量并合并最频繁词对的过程，会持续进行一定数量的步骤，或者直到达到目标词汇量

汉字转拼音 ，然后就可以使用转为英文设计的分词器